# AlumniKonneQt · from-scratch engineering pilot

**Machine labels, not human-reviewed.** This notebook fits random character-CNN weights and checks validation results. It does not evaluate the test pool or replace the app model. The tiny data can reveal pipeline failures; it cannot establish real mentorship quality.

Local execution is supported. In Colab, upload the supplied pilot ZIP once when prompted. No API keys, paid GPU or pretrained model download is required.

## 1. Load the verified bundle
Local: run from the repository root. Colab: upload the pilot bundle, then files are checked against the expected source/data hashes.

In [ ]:
from pathlib import Path
import sys, os, tempfile, zipfile, hashlib, subprocess
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    assert len(archives) == 1, 'Upload exactly one pilot ZIP'
    ROOT = Path(tempfile.mkdtemp(prefix='alumnikonneqt-pilot-'))
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            target = (ROOT / member.filename).resolve()
            assert target.is_relative_to(ROOT.resolve()), 'Invalid archive path'
        archive.extractall(ROOT)
else:
    ROOT = Path.cwd()
    assert (ROOT / 'ml/train_scratch_pilot.py').is_file(), 'Run from the repository root'
DATA = ROOT / 'ml/data/pilot-machine-v1'
EXPECTED_IMPL = 'db933b4687ef36026a9e6a8e8b2caf914ead57485227a137b605f22df94dd393'
EXPECTED_DATA = '3ec6ce8838a7af6624c46a592097f0ac117c70d1286251bb6000b22b1f56ee6b'
assert hashlib.sha256((ROOT / 'ml/train_scratch_pilot.py').read_bytes()).hexdigest() == EXPECTED_IMPL
assert hashlib.sha256((DATA / 'manifest.json').read_bytes()).hexdigest() == EXPECTED_DATA
print('Verified implementation and engineering-pilot manifest.')

## 2. Prepare dependencies and persistent outputs
Colab installs the pinned requirements and asks to mount Drive so outputs survive runtime deletion. Locally the isolated environment is already prepared. The upload/mount branch requires your Google account and has not been exercised by local execution.

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'ml/requirements-pilot.txt')], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/AlumniKonneQt-ML/runs')
else:
    RUN_ROOT = ROOT / 'ml/runs'
os.environ['MPLCONFIGDIR'] = tempfile.mkdtemp(prefix='alumni-mpl-')
from datetime import datetime, timezone
RUN_DIR = RUN_ROOT / ('notebook-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
print('Run directory:', RUN_DIR)
print('Training will use a fresh Python process to avoid stale notebook imports.')

## 3. Train and compare on validation
Vocabulary and TF-IDF fit on training profiles only. Random embeddings feed a small character convolution and directional 64-value projection heads. Cosine predictions fit grade/3 using MSE. Batch 16, learning rate 0.001, at most 10 epochs; early stopping patience 3. The lowest validation loss selects the checkpoint; the three lowest-loss checkpoints are retained. This is a compact pilot, not an established best architecture.

The explicit opt-in below permits unreviewed machine labels for engineering only. No test data or test scores are loaded.

In [ ]:
import json
completed = subprocess.run([sys.executable, str(ROOT/'ml/train_scratch_pilot.py'), str(DATA), str(RUN_DIR), '--epochs', '10', '--allow-machine-labels'], capture_output=True, text=True)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError('Fresh-process training failed; see output above.')
result = json.loads((RUN_DIR/'run.json').read_text())
assert result['reload_verified'] and result['weight_change_l1'] > 0
assert result['test_performance_accessed'] is False
print('Actual parameter updates and checkpoint reload verified.')
print('Epochs completed:', result['epochs_completed'])
print('Validation TF-IDF:', result['baseline_validation'])
print('Validation neural:', result['neural_validation'])
print('Saved outputs:', RUN_DIR)

## 4. Inspect saved evidence
Training loss is a batch average; validation loss is measured at each epoch end. The curves are diagnostics with different sampling. Worse neural results must be retained. No test or production superiority claim follows from this figure.

In [ ]:
from IPython.display import display, Image
import json
history = json.loads((RUN_DIR/'history.json').read_text())
print('Saved epochs:', len(history))
print('Selected checkpoint:', result['best_checkpoint'])
display(Image(filename=str(RUN_DIR/'loss.png')))
print('Checkpoint and raw validation rankings saved under:', RUN_DIR)

## Next steps
- Human label/template review is deferred, not completed.
- Preserve this machine-labelled pilot as its own version; reviewed labels require a new version and rerun.
- Expand independent people and realistic difficult candidates before treating results as a useful final model.
- Keep the test pool out of settings/checkpoint decisions.
- A Python 64-value encoder is not a drop-in replacement for the app’s 384-value MiniLM ONNX model. Export, parity, corpus re-embedding and role/workflow tests remain separate.
- Checkpoints here support selection/reload; this runner does not implement optimizer/RNG-state resume.
- Code and executed notebook stay local unless explicitly pushed/uploaded.
- References: [PyTorch optimization](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html), [train-only preprocessing](https://scikit-learn.org/stable/common_pitfalls.html).